# Clustering & Dimension Reduction - Product Market Segments (Corrected V3)
**Module owner:** Marina

**RQ2:** Do beauty products cluster into distinct market segments based on
price tier and customer response patterns, or do expressed consumer
preferences cut across price tiers?

This version uses the canonical `amazon_beauty_priced_subset_2015plus.csv`
and computes the NLP measures from the real review text. It does not use
synthetic NLP data and does not attempt an unsafe `parent_asin` to `asin`
merge.

**Unit of analysis:** product (`asin`). Reviews are first scored, then
aggregated to one product-level row before clustering.

In [ ]:
from pathlib import Path
from itertools import combinations
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from nltk.sentiment import SentimentIntensityAnalyzer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
MIN_REVIEWS = 5
EARLY_WINDOW = 5
# Keep the cleaned priced dataset in the same folder as this notebook.
DATA_PATH = Path("amazon_beauty_priced_subset_2015plus.csv")
OUTPUT_DIR = Path(".")
CLUSTER_OUTPUT = OUTPUT_DIR / "product_clusters_k4_2015plus_v3.csv"
FIGURE_OUTPUT = OUTPUT_DIR / "pca_clusters_v3.png"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Required input file not found: {DATA_PATH}. "
        "Place amazon_beauty_priced_subset_2015plus.csv in the same "
        "folder as this notebook."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Libraries loaded")
print(f"Data source: {DATA_PATH.resolve()}")
print(f"Output folder: {OUTPUT_DIR.resolve()}")


## 1. Load and validate the canonical priced data

The cleaning notebook already applied the inclusive 2015 cutoff,
deduplication, timestamp conversion, metadata merge, and price filter.

In [ ]:
usecols = [
    "asin", "rating", "text", "helpful_vote", "verified_purchase",
    "review_date", "review_year", "review_length", "review_wordcount",
    "has_text", "price", "price_tier", "product_title", "store",
]
df = pd.read_csv(
    DATA_PATH,
    usecols=usecols,
    parse_dates=["review_date"],
    low_memory=False,
)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["helpful_vote"] = pd.to_numeric(
    df["helpful_vote"], errors="coerce"
).fillna(0)
df["verified_purchase"] = (
    df["verified_purchase"].astype(str).str.lower()
    .map({"true": True, "false": False})
)
df = df.dropna(
    subset=["asin", "rating", "price", "review_date"]
).copy()

year_check = df["review_year"].min() >= 2015
price_check = df["price"].notna().all()
print(f"[{'PASS' if year_check else 'WARNING'}] Earliest review year is 2015 or later.")
print(f"[{'PASS' if price_check else 'WARNING'}] All retained reviews have a valid price.")
print(f"Loaded: {len(df):,} reviews, {df['asin'].nunique():,} ASINs")
print(f"Date range: {df['review_date'].min()} to {df['review_date'].max()}")

## 2. Score the real review text using the team NLP definitions

This reproduces the attached NLP notebook's VADER compound score,
rating-sentiment discrepancy, hidden-dissatisfaction definition, and
early-review logic on the same 2015+ population used by clustering.

In [ ]:
def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).lower()
    value = re.sub(r"http\S+|www\.\S+", " ", value)
    value = re.sub(r"[^a-z0-9\s']", " ", value)
    return re.sub(r"\s+", " ", value).strip()

sia = SentimentIntensityAnalyzer()
df["clean_text"] = df["text"].map(clean_text)
df["sentiment_compound"] = df["clean_text"].map(
    lambda text: sia.polarity_scores(text)["compound"] if text else 0.0
)
df["scaled_rating"] = (df["rating"] - 3.0) / 2.0
df["discrepancy"] = df["sentiment_compound"] - df["scaled_rating"]
df["hidden_dissatisfaction"] = (
    df["rating"].ge(4) & df["sentiment_compound"].lt(-0.05)
)

df = df.sort_values(["asin", "review_date"]).copy()
df["review_rank"] = df.groupby("asin").cumcount() + 1
df["early_sentiment"] = df["sentiment_compound"].where(
    df["review_rank"].le(EARLY_WINDOW)
)

high_rating = df["rating"].ge(4)
print(
    "Hidden dissatisfaction among 4-5-star reviews: "
    f"{df.loc[high_rating, 'hidden_dissatisfaction'].mean():.2%}"
)

## 3. Aggregate reviews to one row per product

Products with at least five retained reviews are included. Review velocity
uses a minimum one-month exposure window and is winsorized at the 99th
percentile to prevent short-lived products from dominating K-means.

In [ ]:
counts = df.groupby("asin").size()
keep_asins = counts[counts.ge(MIN_REVIEWS)].index
analysis_reviews = df[df["asin"].isin(keep_asins)].copy()

agg = analysis_reviews.groupby("asin").agg(
    price=("price", "median"),
    price_tier=("price_tier", "first"),
    product_title=("product_title", "first"),
    store=("store", "first"),
    n_reviews=("rating", "size"),
    avg_rating=("rating", "mean"),
    rating_std=("rating", "std"),
    pct_verified=("verified_purchase", "mean"),
    avg_helpful=("helpful_vote", "mean"),
    avg_review_len=("review_length", "mean"),
    pct_has_text=("has_text", "mean"),
    first_review=("review_date", "min"),
    last_review=("review_date", "max"),
    avg_sentiment=("sentiment_compound", "mean"),
    sentiment_std=("sentiment_compound", "std"),
    avg_discrepancy=("discrepancy", "mean"),
    pct_hidden_dissatisfaction=("hidden_dissatisfaction", "mean"),
    early_avg_sentiment=("early_sentiment", "mean"),
).reset_index()

agg["rating_std"] = agg["rating_std"].fillna(0)
agg["sentiment_std"] = agg["sentiment_std"].fillna(0)
agg["span_days"] = (agg["last_review"] - agg["first_review"]).dt.days
agg["active_months"] = (agg["span_days"] / 30.44).clip(lower=1.0)
agg["reviews_per_month_raw"] = agg["n_reviews"] / agg["active_months"]
velocity_cap = agg["reviews_per_month_raw"].quantile(0.99)
agg["reviews_per_month"] = agg["reviews_per_month_raw"].clip(
    upper=velocity_cap
)

print(f"Eligible products: {len(agg):,}")
print(
    f"Reviews represented: {len(analysis_reviews):,} "
    f"({len(analysis_reviews) / len(df):.1%})"
)
print(f"Review-velocity 99th-percentile cap: {velocity_cap:.2f}")

## 4. Transform and standardize clustering features

Highly correlated word count is omitted because review character length
already measures review detail. Skewed nonnegative variables are
log-transformed before standardization.

In [ ]:
features = [
    "price", "avg_rating", "rating_std", "pct_verified",
    "avg_helpful", "avg_review_len", "pct_has_text",
    "reviews_per_month", "avg_sentiment", "sentiment_std",
    "avg_discrepancy", "pct_hidden_dissatisfaction",
    "early_avg_sentiment",
]
log_features = [
    "price", "avg_helpful", "avg_review_len", "reviews_per_month"
]

X = agg[features].copy()
for column in log_features:
    X[column] = np.log1p(X[column].clip(lower=0))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
finite_feature_check = np.isfinite(X_scaled).all()
print(f"[{'PASS' if finite_feature_check else 'WARNING'}] All scaled feature values are finite.")
print(f"Feature matrix: {X_scaled.shape}")

## 5. Compare candidate K-means solutions

In [ ]:
diagnostic_rows = []
for k in range(2, 9):
    model = KMeans(
        n_clusters=k, random_state=RANDOM_STATE,
        n_init=20, max_iter=500,
    )
    labels = model.fit_predict(X_scaled)
    diagnostic_rows.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(
                X_scaled, labels,
                sample_size=min(5000, len(X_scaled)),
                random_state=RANDOM_STATE,
            ),
            "calinski_harabasz": calinski_harabasz_score(
                X_scaled, labels
            ),
            "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        }
    )

diagnostics = pd.DataFrame(diagnostic_rows)
diagnostics.round(4)

**Choice: K=4.** K is retained as a substantively motivated segmentation,
not claimed to be the unique statistical optimum. The candidate metrics
are reported transparently. K=4 allows the analysis to isolate a
behaviorally distinctive high-engagement, low-verification group while
maintaining stable and interpretable broader market segments.

In [ ]:
K = 4
kmeans = KMeans(
    n_clusters=K, random_state=RANDOM_STATE,
    n_init=50, max_iter=500,
)
agg["cluster"] = kmeans.fit_predict(X_scaled)

seed_labels = [
    KMeans(
        n_clusters=K, random_state=seed,
        n_init=20, max_iter=500,
    ).fit_predict(X_scaled)
    for seed in range(10)
]
stability_ari = [
    adjusted_rand_score(seed_labels[i], seed_labels[j])
    for i, j in combinations(range(10), 2)
]
print(
    f"K=4 stability: mean ARI={np.mean(stability_ari):.3f}, "
    f"minimum ARI={np.min(stability_ari):.3f}"
)
print(agg["cluster"].value_counts().sort_index())

## 6. Profile and name the four clusters

In [ ]:
profile_features = features + ["n_reviews"]
profile = agg.groupby("cluster")[profile_features].mean()
profile["n_products"] = agg["cluster"].value_counts().sort_index()

# Names are assigned from reproducible profile rankings, not cluster IDs.
low_verification = profile["pct_verified"].idxmin()
lowest_satisfaction = profile["avg_rating"].idxmin()
highest_satisfaction = profile["avg_rating"].idxmax()
remaining = (
    set(profile.index)
    - {low_verification, lowest_satisfaction, highest_satisfaction}
)

cluster_names = {
    low_verification: "High-Engagement / Low-Verification",
    lowest_satisfaction: "Budget / Polarizing",
    highest_satisfaction: "Quiet Reliable Winners",
}
for cluster in remaining:
    cluster_names[cluster] = "High-Rating / Sentiment-Mismatch"

agg["cluster_name"] = agg["cluster"].map(cluster_names)
profile["cluster_name"] = profile.index.map(cluster_names)
profile.round(3)

**Interpretation caution for the low-verification segment:** verified
purchase status is an imperfect behavioral indicator. A low verified
share does not establish fraudulent, manipulated, or inauthentic reviews.
The segment should be described as behaviorally distinctive, and its
sentiment pattern should be reported separately.

## 7. PCA dimensionality reduction and final figure

In [ ]:
pca = PCA(random_state=RANDOM_STATE)
X_pca_full = pca.fit_transform(X_scaled)
agg["PC1"] = X_pca_full[:, 0]
agg["PC2"] = X_pca_full[:, 1]

cumulative = np.cumsum(pca.explained_variance_ratio_)
n_components_90 = int(np.searchsorted(cumulative, .90) + 1)
print(
    f"PC1 + PC2 explained variance: "
    f"{pca.explained_variance_ratio_[:2].sum():.1%}"
)
print(f"Components needed for >=90% variance: {n_components_90}")

In [ ]:
tier_order = [
    "Budget(<$10)", "Mid($10-25)", "Premium($25-50)",
    "Luxury($50-100)", "Ultra(>$100)",
]
agg["price_tier"] = pd.Categorical(
    agg["price_tier"], categories=tier_order, ordered=True
)

cluster_colors = {
    "High-Rating / Sentiment-Mismatch": "#4C78A8",
    "Quiet Reliable Winners": "#59A14F",
    "High-Engagement / Low-Verification": "#F28E2B",
    "Budget / Polarizing": "#E15759",
}
tier_colors = ["#482475", "#355F8D", "#21918C", "#73D055", "#FDE725"]
tier_cmap = ListedColormap(tier_colors)
tier_norm = BoundaryNorm(np.arange(-.5, 5.5, 1), tier_cmap.N)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name in [
    "High-Rating / Sentiment-Mismatch", "Quiet Reliable Winners",
    "High-Engagement / Low-Verification", "Budget / Polarizing",
]:
    subset = agg[agg["cluster_name"].eq(name)]
    axes[0].scatter(
        subset["PC1"], subset["PC2"],
        label=name, color=cluster_colors[name],
        s=10, alpha=.58, edgecolors="none",
    )
axes[0].set_title("Products in PCA Space, Colored by Cluster")
axes[0].set_xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)"
)
axes[0].set_ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)"
)
axes[0].legend(title="Market segment", fontsize=8, loc="best")

tier_codes = agg["price_tier"].cat.codes
scatter = axes[1].scatter(
    agg["PC1"], agg["PC2"], c=tier_codes,
    cmap=tier_cmap, norm=tier_norm,
    s=10, alpha=.58, edgecolors="none",
)
axes[1].set_title("Products in PCA Space, Colored by Price Tier")
axes[1].set_xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)"
)
axes[1].set_ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)"
)
colorbar = fig.colorbar(
    scatter, ax=axes[1], ticks=range(5), pad=.02
)
colorbar.ax.set_yticklabels(tier_order, fontsize=8)

fig.suptitle(
    "Amazon Beauty Product Segments (2015+ Real Review Sentiment)",
    fontsize=15, y=1.01,
)
plt.tight_layout()
plt.savefig(FIGURE_OUTPUT, dpi=200, bbox_inches="tight")
plt.show()

## 8. Save the corrected product-level output

In [ ]:
output_columns = [
    "asin", "product_title", "store", "price", "price_tier",
    "n_reviews", "avg_rating", "rating_std", "pct_verified",
    "avg_helpful", "avg_review_len", "pct_has_text",
    "reviews_per_month", "avg_sentiment", "sentiment_std",
    "avg_discrepancy", "pct_hidden_dissatisfaction",
    "early_avg_sentiment", "cluster", "cluster_name", "PC1", "PC2",
]
agg[output_columns].to_csv(CLUSTER_OUTPUT, index=False)
print(f"Saved: {CLUSTER_OUTPUT}")
print(f"Saved: {FIGURE_OUTPUT}")

## Report-ready conclusion

The analysis identified four stable, overlapping product segments based on
price, rating behavior, review engagement, purchase verification, and
real-text sentiment. One segment combined high ratings with substantially
weaker text sentiment and elevated hidden dissatisfaction, demonstrating
why ratings alone do not fully represent customer response. Price tier
influenced segment membership but did not fully determine it, because
products from multiple price tiers appeared within every cluster. The
high-engagement, low-verification segment was behaviorally distinctive,
but its low verified-purchase share should not be interpreted as evidence
of inauthentic reviews. Overall, the modest cluster-separation metrics
indicate soft market segments rather than sharply separated natural
categories.